# 2. Pipeline Parallelism

Pipeline parallelism splits the model into **stages**.

**Core idea**
- GPU 0 runs the early layers
- GPU 1 runs the later layers
- the batch is broken into micro-batches
- each micro-batch flows through the stages like an assembly line

This helps when a single GPU cannot hold the whole model.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Beginner mental model

Think of a two-stage factory:
- Stage 1 prepares features
- Stage 2 turns those features into predictions

We will:
1. split a tiny model into two stages
2. run the batch in micro-batches
3. verify that the pipelined result matches the full model

In [ ]:
from copy import deepcopy


class Stage1(nn.Module):
    """First half of the model."""

    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.linear = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.linear(x))


class Stage2(nn.Module):
    """Second half of the model."""

    def __init__(self, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


class TinyPipelineModel(nn.Module):
    """Convenient full model for comparison."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        self.stage1 = Stage1(input_dim, hidden_dim)
        self.stage2 = Stage2(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.stage2(self.stage1(x))


input_dim = 6
hidden_dim = 8
output_dim = 2
batch_size = 8
micro_batch_size = 2
learning_rate = 0.05

x = torch.randn(batch_size, input_dim)
y = torch.randn(batch_size, output_dim)
criterion = nn.MSELoss(reduction="sum")

base_model = TinyPipelineModel(input_dim, hidden_dim, output_dim)
base_state = deepcopy(base_model.state_dict())

print("Batch shape:", tuple(x.shape))
print("Micro-batches:", len(x) // micro_batch_size)

In [ ]:
full_model = TinyPipelineModel(input_dim, hidden_dim, output_dim)
full_model.load_state_dict(base_state)
full_output = full_model(x)

stage1 = deepcopy(full_model.stage1)
stage2 = deepcopy(full_model.stage2)

pipeline_outputs = []
for micro_batch_id, x_micro in enumerate(x.split(micro_batch_size)):
    hidden = stage1(x_micro)
    output = stage2(hidden)
    pipeline_outputs.append(output)
    print(
        f"Micro-batch {micro_batch_id}: "
        f"{tuple(x_micro.shape)} -> {tuple(hidden.shape)} -> {tuple(output.shape)}"
    )

pipeline_output = torch.cat(pipeline_outputs, dim=0)

print()
print("Max forward difference:", (full_output - pipeline_output).abs().max().item())
assert torch.allclose(full_output, pipeline_output, atol=1e-7)
print("Forward pass matches the unsplit model.")

In [ ]:
def train_full_model(initial_state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    model = TinyPipelineModel(input_dim, hidden_dim, output_dim)
    model.load_state_dict(initial_state)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    optimizer.zero_grad()
    loss = criterion(model(x), y) / batch_size
    loss.backward()
    optimizer.step()
    return deepcopy(model.state_dict())


def train_pipeline_model(initial_state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    model = TinyPipelineModel(input_dim, hidden_dim, output_dim)
    model.load_state_dict(initial_state)
    stage1 = deepcopy(model.stage1)
    stage2 = deepcopy(model.stage2)
    optimizer = torch.optim.SGD(
        list(stage1.parameters()) + list(stage2.parameters()),
        lr=learning_rate,
    )

    optimizer.zero_grad()

    for x_micro, y_micro in zip(x.split(micro_batch_size), y.split(micro_batch_size)):
        prediction = stage2(stage1(x_micro))
        micro_loss = criterion(prediction, y_micro)
        micro_loss.backward()

    for parameter in list(stage1.parameters()) + list(stage2.parameters()):
        parameter.grad /= batch_size

    optimizer.step()

    merged = TinyPipelineModel(input_dim, hidden_dim, output_dim)
    merged.stage1.load_state_dict(stage1.state_dict())
    merged.stage2.load_state_dict(stage2.state_dict())
    return deepcopy(merged.state_dict())


full_state = train_full_model(base_state)
pipeline_state = train_pipeline_model(base_state)

max_difference = max(
    (full_state[name] - pipeline_state[name]).abs().max().item()
    for name in full_state
)

print("Max parameter difference after one step:", max_difference)
assert max_difference < 1e-7
print("Pipeline training matches the unsplit training step.")

## Takeaways

- Pipeline parallelism splits the model by **layers or blocks**.
- Micro-batches keep later stages busy instead of waiting for the
  whole batch.
- The main tradeoff is extra scheduling complexity and possible
  idle time, sometimes called a **pipeline bubble**.